# Author a fault-tolerant chemistry PREPARE operation

A block encoding of the second-quantized Hamiltonian uses PREPARE to
load coefficient magnitudes, SELECT to apply the chosen term, and the
inverse of PREPARE to close the block. Begin with a declaration, estimate
the lookup scaling, then inspect explicit alias-table wiring and an
executable sparse path.

Adapted from the [Qualtran v0.7.0 tutorial](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/chemistry/writing_algorithms.ipynb), Copyright 2023 Google LLC, Apache-2.0.

## Start with the selection registers

Write down the four spatial-orbital words p, q, r, s and the spin and
sign bits before modeling work registers. The first sketch assumes four
uniform preparations, one lookup, one comparator and four controlled
swaps. Approximate controlled swaps are accounted for as CCZ equivalents
in this tutorial's budget.

In [ ]:
from io import BytesIO

from IPython.display import Image, SVG, display
import matplotlib.pyplot as plt
import numpy as np

from examples.algorithms.chemistry_authoring import (
    DensePreparationSketch, authoring_integrals, dense_alias_rows, tutorial_ccz_equivalents,
)
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot


def show_dot(dot):
    display(SVG(data=render_call_graph_svg(dot).decode()))


def show_figure(figure):
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png")
        plt.close(figure)
        display(Image(data=buffer.getvalue()))


show_dot(export_signature_graph_dot(DensePreparationSketch(4, 16).signature(), "Dense PREPARE sketch"))

## Compare direct and blocked lookup

The dense table has one-body and two-body rows. Direct lookup scales
with the table size. Blocked lookup splits the address into a high part
and an in-block part, trading gate cost for additional storage. The
fits summarize the sampled sizes.

In [ ]:
def scaling_fit(axis, bases, block, label):
    budgets = np.asarray([
        tutorial_ccz_equivalents(DensePreparationSketch(int(n), 16).cost(block))
        for n in bases
    ])
    power, intercept = np.polyfit(np.log(bases), np.log(budgets), 1)
    points = axis.loglog(bases, budgets, "o", label=f"{label}, slope {power:.2f}")
    dense = np.linspace(bases[0], bases[-1], 200)
    axis.loglog(dense, np.exp(intercept) * dense**power, "--", color=points[0].get_color())
    return power


fig, axis = plt.subplots(figsize=(6.5, 4), layout="constrained")
direct_slope = scaling_fit(axis, np.linspace(10, 100, 10, dtype=int), 1, "Direct QROM")
assert 3.8 < direct_slope < 4.2
axis.set(xlabel="Spin orbitals", ylabel="CCZ equivalents")
axis.legend()
show_figure(fig)

In [ ]:
fig, axis = plt.subplots(figsize=(7.5, 4.5), layout="constrained")
# The original scaling sweep includes odd spin counts and floors n/2.
sizes = np.linspace(50, 200, 10, dtype=int)
for block in (1, 2, 8, 32, None):
    slope = scaling_fit(axis, sizes, block, "Optimal block" if block is None else f"Block {block}")
assert 1.8 < slope < 2.5
axis.set(xlabel="Spin orbitals", ylabel="CCZ equivalents")
axis.legend()
show_figure(fig)

## Add data and wire the detailed graph

The contiguous four-index operation declares the mapping's input and
output widths and a Toffoli budget. The locked tutorial leaves the actual
index arithmetic as an exercise. The detailed score follows the selected
lookup banks, Hadamards on the sampling register, the keep comparison,
four controlled swaps and a final comparison to clear the decision bit.

For a reproducible example, use the source tensor distributions with seed
7. The row conversion uses the flattened one-body table first and the
four-index table second. One-body alternate rows pad the last two
indices with zero.

In [ ]:
from flagquantum.drawer.register_score_data import register_score_data
from flagquantum.drawer.register_score_mpl import draw_register_score

spin_orbitals = 10
one, two = authoring_integrals(spin_orbitals // 2)
detailed = DensePreparationSketch(spin_orbitals, 10)
alias_rows = dense_alias_rows(one, two, detailed.precision)
assert all(len(column) == detailed.rows for column in alias_rows)
show_dot(export_signature_graph_dot(detailed.contiguous_signature(), "ContiguousIndexFourD"))
figure, _ = draw_register_score(register_score_data(detailed.register_score()), max_width=13, max_height=7)
show_figure(figure)

model = detailed.detailed_cost(alias_rows)
show_dot(export_modeled_call_graph_dot(model, max_depth=1))
sketch_budget = tutorial_ccz_equivalents(DensePreparationSketch(spin_orbitals, 16).cost())
detailed_budget = tutorial_ccz_equivalents(model)
assert (sketch_budget, detailed_budget) == (328, 562)
print("16-bit sketch and 10-bit detailed model:", sketch_budget, detailed_budget)

## Compare the two four-bit preparations

Repeat the dense estimate with four alias precision bits. Use only
symmetry representatives in the sparse preparation and build its
native circuit. The detailed dense graph continues to use the opaque
contiguous-index cost. Its estimate therefore has a different scope
from the sparse instruction report.

In [ ]:
from collections import Counter

from flagquantum.algorithms.primitives.sparse_chemistry_data import sparse_chemistry_data
from flagquantum.algorithms.primitives.sparse_chemistry_prepare import PrepareSparseChemistry
from flagquantum.compiler.registers.layout.partition import partition_register
from flagquantum.dynamic import DynamicCircuit

dense = DensePreparationSketch(spin_orbitals, 4)
dense_budget = tutorial_ccz_equivalents(dense.detailed_cost(dense_alias_rows(one, two, 4)))
sparse = PrepareSparseChemistry(sparse_chemistry_data(one, two, probability_bits=4))
sparse_budget = tutorial_ccz_equivalents(sparse.resource_model())
assert (dense_budget, sparse_budget) == (502, 153)

layout = sparse.selection_layout + sparse.junk_layout
logical = sum(width for _, width in layout)
work_width, slots = sparse.requirements()
workspace = tuple(range(logical, logical + work_width))
op = sparse.make_on(
    partition_register(tuple(range(logical)), layout),
    workspace=workspace, classical_bits=tuple(range(slots)),
)
ir = DynamicCircuit(logical + work_width).append(op).to_ir()
counts = Counter(gate.name for gate in ir.instructions)
assert counts["measure"] > 0
print("Four-bit dense and sparse model budgets:", dense_budget, sparse_budget)
print("Native sparse retained rows, scratch, emitted measurements:",
      len(sparse.data.indices), work_width, counts["measure"])